# A Simple Compiler for a Fragment of  `C`

This file shows how a simple compiler for a fragment of the programming language `C` can be implemented using `Larkita`.  In contrast to the notebook `Compiler.ipynb`, this fragment also
contains a `for` loop of the form `for (i = a .. b) stmnt`, which executes the statement `stmnt` for all values
of the variable `i` from `a` up to `b`.

First, we describe the grammar of our language using EBNF-notation, since this enables us to give a very concise presentation of the grammar.
Below, the token 'NUMBER' stands for natural numbers and 'ID' represents identifiers.
```
program: function+
       
function: 'int' ID '(' ('int' ID (',' 'int' ID)*)?  ')' '{' decl* stmnt* '}' 
    
decl: 'int' ID ';'  
    
stmnt: '{' stmnt* '}'
     | ID '=' expr ';' 
     | 'if' '(' boolExpr ')' stmnt 
     | 'if' '(' boolExpr ')' stmnt 'else' stmnt 
     | 'while' '(' boolExpr ')' stmnt 
     | 'for' '(' ID '=' expr '..' expr ')' stmnt
     | 'return' expr ';'
     | expr ';'             
     
boolExpr: boolExpr ('&&' | '||') boolExpr
        | '!' boolExpr             
        | '(' boolExpr ')'
        | expr ('==' | '!=' | '<=' | '>=' | '<' | '>')  expr 
     
expr: expr ('+' | '-' | '*' | '/' | '%') expr 
    | '(' expr ')' 
    | NUMBER
    | ID ('(' (expr (',' expr)*)? ')')?  
```
This grammar is ambiguous.  In the version of this notebook that is based on `Ply` the ambiguity had been resolved
by specifying the appropriate operator precedences.  `Larkita` offers no precedence declarations for its 
<span style="font-variant: small-caps;">LR</span> parser.  Therefore, we will instead *stratify* the grammar:  we
introduce one syntactical variable per precedence level.  This removes the ambiguity and, as a side effect, makes the
precedences and associativities visible in the grammar itself.

## Specification of the Scanner

Unlike `Ply`, which separates the scanner (`ply.lex`) from the parser (`ply.yacc`), `Larkita` reads a *single* grammar
that specifies both the *tokens* and the *grammar rules*:
- The names of *terminals*, i.e. of the tokens, are written in `UPPER_CASE`.  A terminal is defined either by a
  string enclosed in double quotes or by a regular expression enclosed in forward slashes.
- The names of *syntactical variables* are written in `lower_case`.

We build this grammar as a string.  In this section we collect those parts of the grammar that describe the scanner
in the variable `terminal_definitions`, while the next section collects the grammar rules in the variable `syntax`.
At the end, both strings are concatenated.

In [ ]:
from larkita import Larkita, UnexpectedInput

The token `NUMBER` specifies a natural number.  Note that the regular expression is enclosed in forward slashes.

In [ ]:
terminal_definitions = r"""
NUMBER: /0|[1-9][0-9]*/
"""

Next, we define the tokens for those operator symbols that consist of more than one character.  Strictly
speaking, these definitions are not necessary:  In `Larkita`, a string that occurs literally inside a grammar rule,
for example `"=="`, is turned into an *anonymous terminal* automatically.  However, an anonymous terminal that is
built from a symbol like `"=="` receives a generated name of the form `__ANON_1`, and such a name is of little help
when we inspect the token stream or when we read an error message.  If a terminal with the same definition has been
declared explicitly, `Larkita` uses this terminal instead of creating an anonymous one.  Therefore, the declarations
below only serve to give these tokens readable names.

Note that a terminal can also be defined by a string.  In this case no escaping is needed:  in contrast to the
`Ply` version, where the token `t_OR` had to be defined by the regular expression `r'\|\|'`, we can simply write
`"||"`.  Similarly, the token `t_DOTDOT = r'\.\.'` becomes `".."`.  Furthermore, `Larkita` sorts string terminals by their length, so `"!="` is preferred over `"!"` and `"<="` is
preferred over `"<"`.

The operators that consist of a single character need no declaration at all:  the list
```
literals = ['+', '-', '*', '/', '%', '(', ')', '{', '}', ';', '=', '<', '>', '!', ',']
```
that was needed for `Ply` has no counterpart in `Larkita`, since `Larkita` names the corresponding anonymous terminals
`PLUS`, `MINUS`, `STAR`, `LPAR`, `RPAR`, and so on.

In [ ]:
terminal_definitions = terminal_definitions + r"""
EQ : "=="
NE : "!="
LE : "<="
GE : ">="
AND: "&&"
OR : "||"
DOTDOT: ".."
"""

Our version of `C` allows both *single-line comments* and *multi-line comments*.
- The regular expression `/\*(.|\n)*?\*/` recognizes multi-line comments.
  Multi-line comments start with the string `/*` and end with the string `*/`.
  Note the use of the *non-greedy* quantor `*?`.  If we have code like
  ```
  /* blah */ a = 1; /* blub */
  ```
  the greedy quantor would recognize the whole line as one comment. 
- The regular expression `//.*` recognizes single-line comments.
  A single line comment starts with the string `//` and extends to the end of the line.

A terminal can be defined as the alternative of several regular expressions.  As the forward slash is the
delimiter of a regular expression in `Larkita`, every occurrence of the character `/` inside a regular expression has
to be escaped as `\/`.

The directive `%ignore` tells the scanner to discard all tokens of the given type, i.e. comments never show up in
the token stream.  This is the `Larkita` equivalent of the function `t_COMMENT` that returned no token.

In [ ]:
terminal_definitions = terminal_definitions + r"""
COMMENT: /\/\*(.|\n)*?\*\// 
       | /\/\/[^\n]*/

%ignore COMMENT
"""

The keywords `'int'`, `'if'`, `'else'`, `'while'`, `'for'`, and `'return'` are syntactically identical to identifiers.
In the `Ply` version of this notebook this problem had been solved via the dictionary
```
Keywords = { 'int'   : 'INT', 
             'if'    : 'IF',
             'else'  : 'ELSE', 
             'while' : 'WHILE', 
             'for'   : 'FOR',
             'return': 'RETURN'
           }
```
and the function `t_ID` that looked up every identifier in this dictionary.

`Larkita` performs this test automatically:  If a string terminal like `"if"` is also matched by the regular expression
of another terminal, in our case by the regular expression defining `NAME`, then `Larkita` first scans a `NAME` and
afterwards checks whether the resulting string is one of the keywords.  If it is, the type of the token is changed
accordingly.  This mechanism is known as the *unless* mechanism.  Because of this, the string `iffy` is scanned as a
single `NAME` and not as the keyword `if` followed by the identifier `fy`.

Therefore, all we have to specify is the terminal `NAME`.  The keywords themselves are written as strings inside the
grammar rules of the next section.

In [ ]:
terminal_definitions = terminal_definitions + r"""
NAME: /[a-zA-Z][a-zA-Z0-9_]*/
"""

White space, i.e. *space characters*, *tabulators*, *carriage returns*, and *newlines* are ignored.
In `Ply` we had needed two declarations for this:  the string `t_ignore` for the first three characters and the
function `t_newline`, which additionally had to increment the line number of the scanner.  In `Larkita`, a single
`%ignore` directive suffices, since `Larkita` keeps track of line numbers and column numbers on its own:  every token
`t` provides the attributes `t.line`, `t.column`, `t.end_line`, and `t.end_column`.  Hence there is no need for a
function like `find_column` either.

In [ ]:
terminal_definitions = terminal_definitions + r"""
%ignore /[ \t\r\n]+/
"""

This is the complete specification of our scanner.

In [ ]:
print(terminal_definitions)

## Specification of the Parser

Since `Ply` does not support <span style="font-variant: small-caps;">Ebnf</span>, the grammar had to be
rewritten:  every list had to be expressed by a pair of recursive rules like `param_list` and `ne_param_list`.
`Larkita` supports <span style="font-variant: small-caps;">Ebnf</span> directly, so we can use the quantifiers `*`,
`+`, and `?` as well as parentheses for grouping.  Consequently, the grammar given at the beginning of this notebook
can be used nearly unchanged.  There are only two things that we have to take care of:
1. The rules for `expr` and `bool_expr` are ambiguous.  Since the 
   <span style="font-variant: small-caps;">LR</span> parser of `Larkita` supports no precedence declarations,
   we have to encode the operator precedences into the grammar by introducing one syntactical variable per
   precedence level.
2. The grammar determines the abstract syntax tree that the parser returns:  The string literals of an
   alternative form the *head* of the tuple that is built for this alternative, while all other symbols become
   its children.  We choose the literals that are ignored and the *aliases* `-> name`, which replace the head,
   in such a way that the parser returns the nested tuples that are used by our code generator.

A program is a non-empty list of function definitions.  The alias `-> "."` makes the parser return a program as a
tuple of the form `('.', function, ...)`, while the alias `-> "fct"` yields function definitions of the form
`('fct', name, parameters, declarations, statements)`:  The string literals `"int"`, `"("`, `")"`, `"{"`, and `"}"`
would only contribute to the head of the tuple, but the alias replaces this head.

The directive `%ignore: "(", ")", "{", "}", ";"` (note the colon) removes the parentheses, the braces, and the
semicolon from the heads of *all* tuples, since these literals carry no information.

In [ ]:
syntax = r"""
program: function+                                                      -> "."

function: "int" NAME "(" parameters ")" "{" declarations statements "}" -> "fct"

%ignore: "(", ")", "{", "}", ";"
"""

The syntactical variables `parameters`, `declarations`, and `statements` describe lists.  In `Ply` these lists
had to be defined via recursion.  Using <span style="font-variant: small-caps;">Ebnf</span>, one rule per list
suffices.  The children of a repetition are spliced into the tuple of the rule.  Hence the alias `-> "."` yields tuples of
the form `('.', name, ...)` and `('.', statement, ...)`:  Note that the tuples of `parameters` and `declarations` contain
only the tokens of type `NAME`, since all other symbols of these rules are string literals, which only contribute to
the head.

In [ ]:
syntax = syntax + r"""
parameters  : ("int" NAME ("," "int" NAME)*)?    -> "."

declarations: ("int" NAME ";")*                  -> "."

statements  : statement*                         -> "."
"""

The rule `statement` has eight alternatives.  As the parentheses, the braces, and the semicolon are ignored,
the heads of the tuples are the keywords `'if'`, `'while'`, and `'return'` and the operator `'='`.  The alternative
for the `if`-`else` statement gets the alias `-> "if-else"`, since otherwise its head would be the tuple
`('if', 'else')`.  Similarly, the alias `-> "for"` replaces the head `('for', '=', '..')` of the `for` loop.  A block `"{" statements "}"` and an expression statement `expr ";"` consist of a single
symbol, as their literals are ignored.  Therefore, they return the tuple of this symbol.

The two alternatives `if_stmnt` and `if_else_stmnt` cause the well-known 
[dangling else problem](https://en.wikipedia.org/wiki/Dangling_else):  when the parser has read a statement of the
form `if ( bool_expr ) statement` and the next token is the keyword `else`, then the parser can either *reduce* this
statement or it can *shift* the token `else`.  In the `Ply` version of this notebook this *shift/reduce conflict*
had been resolved by declaring the precedence of the token `ELSE` to be higher than the precedence of the token
`IF` and by attaching the precedence of `IF` to the rule `stmnt : IF '(' bool_expr ')' stmnt` via the keyword
`%prec`.  

`Larkita` resolves a shift/reduce conflict *in favour of shifting*.  This is exactly what we want:  the keyword `else`
is attached to the innermost `if` that does not yet have an `else` part.  Hence a program of the form
```
if (a == 1) if (b == 1) x = 2; else x = 3;
```
is interpreted as
```
if (a == 1) {
    if (b == 1) {
        x = 2;
    } else {
        x = 3;
    }
}
```
and not as
```
if (a == 1) {
    if (b == 1) {
        x = 2;
    } 
} else {
    x = 3;
}
```

In [ ]:
syntax = syntax + r"""
statement: "{" statements "}"
         | NAME "=" expr ";"
         | "if" "(" bool_expr ")" statement
         | "if" "(" bool_expr ")" statement "else" statement   -> "if-else"
         | "while" "(" bool_expr ")" statement
         | "for" "(" NAME "=" expr ".." expr ")" statement   -> "for"
         | "return" expr ";"
         | expr ";"
"""

The `Ply` version of this notebook had used the precedence declarations
```
('left'    , 'OR'),
('left'    , 'AND'),
('right'   , '!'),
('nonassoc', 'EQ', 'NE', 'LE', 'GE', '<', '>'),
```
to disambiguate the rules for `bool_expr`.  We express the same information by *stratifying* the grammar:
- `bool_expr` is a list of `conjunction`s that are separated by the operator `||`,
- a `conjunction` is a list of `negation`s that are separated by the operator `&&`,
- a `negation` is a `bool_atom` that is preceded by an arbitrary number of `!` operators,
- a `bool_atom` is either a comparison of two arithmetical expressions or a `bool_expr` in parentheses.

Since the operators `||` and `&&` are *left associative*, the corresponding rules are *left recursive*:  the
syntactical variable on the left hand side of the rule occurs as the *first* symbol on the right hand side.  The
operator `!` is *right associative*, hence the rule `not_expr` is *right recursive*.  As the comparison operators
are non-associative, the rule `bool_atom` is not recursive at all:  an expression like `a < b < c` is a syntax
error.

The operators are the heads of the tuples, for example the alternative `bool_expr "||" conjunction` yields the
tuple `('||', lhs, rhs)`.  An alternative that consists of a single symbol, like `bool_expr: conjunction`, returns
the tuple of this symbol.  Hence the tuple of the expression `x == 1` is `('==', x, 1)`, where `x` and `1` stand for
the tuples of the variable and of the number, and not a chain of four nested tuples.

In [ ]:
syntax = syntax + r"""
bool_expr: bool_expr "||" conjunction
         | conjunction

conjunction: conjunction "&&" negation
           | negation

negation: "!" negation
        | bool_atom

bool_atom: expr "==" expr
         | expr "!=" expr
         | expr "<=" expr
         | expr ">=" expr
         | expr "<"  expr
         | expr ">"  expr
         | "(" bool_expr ")"
"""

The arithmetical expressions are stratified in the same way.  The precedence declarations
```
('left', '+', '-'),
('left', '*', '/', '%')
```
of the `Ply` version are replaced by the three levels `expr`, `product`, and `factor`.  Again, all rules are left
recursive because all arithmetical operators are left associative.

As the parentheses are ignored, the last alternative of the rule `factor` consists of a single symbol.  Therefore, a
parenthesized expression creates no tuple of its own.  A number yields a pair of the form `('NUMBER', digits)` and a
variable yields `('NAME', name)`.  The arguments of a function call are written inline as `[expr ("," expr)*]`:
The children of an optional part and of a repetition are spliced into the tuple of the alternative, so a function
call yields the tuple `('call', ('NAME', name), arg_1, ..., arg_n)`.

In [ ]:
syntax = syntax + r"""
expr: expr "+" product
    | expr "-" product
    | product

product: product "*" factor
       | product "/" factor
       | product "%" factor
       | factor

factor: NUMBER
      | NAME
      | NAME "(" [expr ("," expr)*] ")"   -> "call"
      | "(" expr ")"
"""

The complete grammar is the concatenation of the grammar rules and the terminal definitions.

In [ ]:
grammar = syntax + terminal_definitions
print(grammar)

Now we can create the parser.  `Larkita` always generates a *canonical LR(1)* parser.  The keyword argument
`start='program'` declares `program` to be the *start variable* of our grammar.  It could be omitted, since the
start variable defaults to the first rule of the grammar.

`Larkita` writes a description of all states of the LR(1) automaton into the file given by the keyword argument
`out_file`.  This is the `Larkita` equivalent of the file `parser.out` that had been produced by `Ply`.  If there are
shift/reduce conflicts, `Larkita` issues a warning of class `ConflictWarning`.  Reduce/reduce conflicts are errors.

In [ ]:
parser = Larkita(grammar, start='program', out_file='compiler.out')

The only conflict that is reported is the shift/reduce conflict that results from the *dangling else
ambiguity*.  As discussed above, `Larkita` resolves this conflict in favour of shifting and hence in the way that we
want.  All other conflicts have been eliminated by stratifying the grammar.  The following cell prints the section
`Conflicts` of the file `compiler.out`.

In [ ]:
with open('compiler.out') as handle:
    report = handle.read()
print(report[report.index('Conflicts\n'):report.index('States\n')])

## Testing the Scanner

The method `parser.lex` runs the scanner without invoking the parser.  It returns an iterator of tokens.
Every token `t` knows its type `t.type`, its value `t.value`, and its position `t.line` and `t.column`.  Note that
white space and comments do not show up in the resulting list of tokens because of the `%ignore` directives, and
that the keywords have been retyped from `NAME` to `INT`, `IF`, `ELSE`, `WHILE`, `FOR`, and `RETURN`.

In [ ]:
def test_scanner(file_name):
    with open(file_name, 'r') as handle:
        program = handle.read()
    print(program)
    return list(parser.lex(program))

In [ ]:
for t in test_scanner('Examples/PrimesFor.c'):
    print(f'{t.line:>3}, {t.column:>3}: {t.type:<10} {t.value}')

## Construction of the Abstract Syntax Tree

The parser generated by `Larkita` returns the abstract syntax tree as a nested tuple.  We have chosen the grammar in
such a way that these tuples are the same nested tuples that had been computed by the `Ply` version of this notebook:
- a list of *n* items is represented as `('.', item₁, ⋯, itemₙ)`,
- a function definition is represented as `('fct', name, parameters, variables, statements)`,
- a variable is represented as a string, while a number is represented as `('Number', digits)`,
- all other nodes are tuples whose first component is the operator.

There is one exception:  `Larkita` represents a token as a pair consisting of the name of the token and its text,
so a variable is returned as `('NAME', name)` and a number as `('NUMBER', digits)`.  The function `simplify_leaves`
converts these pairs into the representation that is expected by our code generator.

In [ ]:
def simplify_leaves(ast):
    match ast:
        case ('NAME', name):
            return name
        case ('NUMBER', digits):
            return ('Number', digits)
        case (head, *children):
            return (head, *[simplify_leaves(child) for child in children])

The function `parse_program` takes a string containing a `C` program, parses it, and returns the abstract
syntax tree of this program.

If the program contains a syntax error, `Larkita` raises an exception of class `UnexpectedInput`.  In `Ply`, the
corresponding error message had been printed by the function `p_error`.  The exception provides the attributes
`line` and `column`, while the method `get_context` returns the line of the program that contains the error together
with a marker that points at the offending token.  In this case, `parse_program` returns `None`.

In [ ]:
def parse_program(program):
    try:
        ast = parser.parse(program)
    except UnexpectedInput as error:
        print(f'Syntax error in line {error.line}, column {error.column}:')
        print(error.get_context(program))
        return None
    return simplify_leaves(ast)

Below is an example of an erroneous program.

In [ ]:
parse_program('int main() { x = 1 + ; }')

The notebook `AST-2-Dot.ipynb` provides the function `tuple2dot`.  This function can be used to visualize the
abstract syntax tree that is generated by the function `parse_program`.

In [ ]:
%run ../AST2Dot.ipynb

In [ ]:
from typing import TYPE_CHECKING
if TYPE_CHECKING:  # declarations for the type checker, never executed
    import graphviz as gv
    type AST = str | int | float | tuple[str, *tuple["AST", ...]]
    def tuple2dot(t: AST) -> gv.Digraph: ...

The function `parse` takes a `file_name` as its sole argument.  The file is read and parsed.
The resulting abstract syntax tree is visualized using `graphviz`.  In contrast to the `Ply` version, there is no
need to reset a line counter:  `Larkita` computes the line numbers from the string that it is given, hence the line
numbers that occur in error messages are always correct.

In [ ]:
def parse(file_name):
    with open(file_name, 'r') as handle:
        program = handle.read() 
    print(program)
    ast = parse_program(program)
    print(ast)
    if ast is not None:
        return tuple2dot(ast)

In [ ]:
parse('Examples/PrimesFor.c')

## Code Generation

The function `indent` is used to indent the generated assembler commands by preceding them with 8 space characters. 

In [ ]:
def indent(s):
    return ' ' * 8 + s

The method `compile_expr(expr, st, class_name)` takes three arguments:
- `expr` is an *abstract syntax tree* that represents an expression.  
  This abstract syntax tree is in turn represented as a nested tuple.  
- `st` is short for *symbol table*.  This is a dictionary that maps variable
  names to natural numbers.  Given a variable `x`, the number `st[x]` specifies
  the location where the variable `x` is stored on the stack with respect to the 
  local stack frame.
- `class_name` is the name of the class that is to be generated.

The function returns a pair of the form `(cmds, size)`.
- `cmds` is a list of assembler commands,
- `size` is the maximum size of the stack that is needed. 

In [ ]:
def compile_expr(expr, st, class_name):
    match expr:
        case str(var): # expr is a variable name
            Cmd = indent(f'iload {st[var]}')
            return [Cmd], 1
        case 'Number', n:
            Cmd = indent(f'ldc {n}')
            return [Cmd], 1
        case ('+' | '-' | '*' | '/' | '%') as op, lhs, rhs:
            L1, sz1 = compile_expr(lhs, st, class_name)
            L2, sz2 = compile_expr(rhs, st, class_name)
            OpToCmd = { '+': 'iadd', '-': 'isub', '*': 'imul', '/': 'idiv', '%': 'irem' }
            Cmd     = indent(OpToCmd[op])
            return L1 + L2 + [Cmd], max(sz1, 1 + sz2)
        case 'call', 'println', *args:
            CmdLst    = [indent('getstatic java/lang/System/out Ljava/io/PrintStream;')]
            stck_size = 0
            cnt       = 0
            for arg in args:
                L, sz_arg = compile_expr(arg, st, class_name)
                stck_size = max(stck_size, cnt + 1 + sz_arg)
                CmdLst   += L
                cnt      += 1
            CmdLst += [indent(f'invokevirtual java/io/PrintStream/println({"I"*cnt})V')]
            return CmdLst, stck_size
        case 'call', f, *args:
            CmdLst    = []
            stck_size = 0
            cnt       = 0
            for arg in args:
                L, sz_arg = compile_expr(arg, st, class_name)
                stck_size = max(stck_size, cnt + sz_arg)
                CmdLst   += L
                cnt      += 1
            CmdLst += [indent(f'invokestatic {class_name}/{f}({"I"*cnt})I')]
            return CmdLst, max(stck_size, 1)
        case _:
            assert False, f'Error in compile_expr({expr}, {st}, {class_name})'

The following is a test of the function `compile_expr`.

In [ ]:
expr = ('call', 'sum', ('+', 'x', 'y'))
st   = { 'x': 0, 'y': 1 }
compile_expr(expr, st, 'Sum')

In [ ]:
expr = ('+', 'x', 'y')
st   = { 'x': 0, 'y': 1 }
compile_expr(expr, st, 'Sum')

The variable `label_counter` is a global counter that is used to create unique label names.
Every call of `new_label` creates a new, unique label.

In [ ]:
expr = ('call', 'println', 'x', 'y')
st   = { 'x': 0, 'y': 1 }
compile_expr(expr, st, 'Sum')

In [ ]:
label_counter = 0

def new_label():
    global label_counter
    label_counter += 1
    return 'l' + str(label_counter)

The method `compile_bool(expr, st, class_name)` takes three arguments:
- `expr` is an *abstract syntax tree* that represents a Boolean expression.  
  This abstract syntax tree is in turn represented as a nested tuple.  
- `st` is short for *symbol table*.  This is a dictionary that maps variable
  names to natural numbers.  Given a variable `x`, the number `st[x]` specifies
  the location where the variable `x` is stored on the stack with respect to the 
  local stack frame.
- `class_name` is the name of the class that is to be generated.

The function returns a pair of the form `(cmds, size)`.
- `cmds` is a list of assembler commands,
- `size` is the maximum size of the stack that is needed. 

In [ ]:
def compile_bool(expr, st, class_name):
    match expr:
        case ('==' | '!=' | '<=' | '>=' | '<' | '>') as op, lhs, rhs:
            OpToCmd = { '==': 'if_icmpeq', 
                        '!=': 'if_icmpne', 
                        '<=': 'if_icmple',
                        '>=': 'if_icmpge',
                        '<' : 'if_icmplt',
                        '>' : 'if_icmpgt'
                      }
            L1, sz1    = compile_expr(lhs, st, class_name)
            L2, sz2    = compile_expr(rhs, st, class_name)
            true_label = new_label()
            next_label = new_label()
            CmdLst     = L1 + L2
            cmd        = OpToCmd[op]
            CmdLst    += [indent(cmd + ' ' + true_label)]
            CmdLst    += [indent('bipush 0')]
            CmdLst    += [indent('goto ' + next_label)]
            CmdLst    += [' ' * 4 + true_label + ':']
            CmdLst    += [indent('bipush 1')]
            CmdLst    += [' ' * 4 + next_label + ':']
            return CmdLst, max(sz1, 1 + sz2)
        case ('&&' | '||') as op, lhs, rhs:
            OpToCmd      = { '&&': 'iand', '||': 'ior' }
            L1, sz1      = compile_bool(lhs, st, class_name)
            L2, sz2      = compile_bool(rhs, st, class_name)
            cmd          = OpToCmd[op]
            CmdLst       = L1 + L2 + [indent(cmd)]
            return CmdLst, max(sz1, 1 + sz2)
        case '!', arg:
            L, sz  = compile_bool(arg, st, class_name)
            CmdLst = [indent('bipush 1')] + L + [indent('isub')]
            return CmdLst, sz + 1
        case _:
            assert False, f'Error in compile_bool({expr}, {st}, {class_name})'

Below is a test for the function `compile_bool`.

In [ ]:
expr = ('==', 'x', 'y')
st   = { 'x': 0, 'y': 1}
compile_bool(expr, st, 'Sum')

In [ ]:
expr = ('!', ('==', 'x', 'y'))
st   = { 'x': 0, 'y': 1}
compile_bool(expr, st, 'Sum')

In [ ]:
expr = ('||', ('==', 'x', 'y'), ('<', 'x', 'y'))
st   = { 'x': 0, 'y': 1}
compile_bool(expr, st, 'Sum')

The method `compile_stmnt(stmnt, st, class_name)` takes three arguments:
- `stmnt` is an *abstract syntax tree* that represents a statement.  
  This abstract syntax tree is in turn represented as a nested tuple.  
- `st` is short for *symbol table*.  This is a dictionary that maps variable
  names to natural numbers.  Given a variable `x`, the number `st[x]` specifies
  the location where the variable `x` is stored on the stack with respect to the 
  local stack frame.
- `class_name` is the name of the class that is to be generated.

The function returns a pair of the form `(cmds, size)`.
- `cmds` is a list of assembler commands,
- `size` is the maximum size of the stack that is needed. 

In [ ]:
def compile_stmnt(stmnt, st, class_name):
    match stmnt:
        case '=', var, expr:
            CmdLst, sz = compile_expr(expr, st, class_name)
            CmdLst    += [indent(f'istore {st[var]}')]
            return CmdLst, sz
        case 'if', expr, sub_stmnt:
            L1, sz1    = compile_bool(expr, st, class_name)
            L2, sz2    = compile_stmnt(sub_stmnt, st, class_name)
            else_label = new_label()
            lbl_stmnt  = ' ' * 4 + else_label + ':'
            CmdLst = L1 + [indent(f'ifeq {else_label}')] + L2 + [lbl_stmnt]
            return CmdLst, max(sz1, sz2)
        case 'if-else', expr, then_stmnt, else_stmnt:
            L1, sz1    = compile_bool(expr, st, class_name)
            L2, sz2    = compile_stmnt(then_stmnt, st, class_name)
            L3, sz3    = compile_stmnt(else_stmnt, st, class_name)        
            else_label = new_label()
            next_label = new_label()
            if_stmnt   = indent(f'ifeq {else_label}')
            else_stmnt = ' ' * 4 + else_label + ':'
            next_stmnt = ' ' * 4 + next_label + ':'
            goto_stmnt = indent(f'goto {next_label}')
            CmdLst = L1 + [if_stmnt] + L2 + [goto_stmnt, else_stmnt] + L3 + [next_stmnt]
            return CmdLst, max(sz1, sz2, sz3)
        case 'while', expr, body_stmnt:
            L1, sz1    = compile_bool(expr, st, class_name)
            L2, sz2    = compile_stmnt(body_stmnt, st, class_name)
            loop_label = new_label()
            next_label = new_label()
            if_stmnt   = indent(f'ifeq {next_label}')
            loop_stmnt = ' ' * 4 + loop_label + ':'        
            next_stmnt = ' ' * 4 + next_label + ':'
            goto_stmnt = indent(f'goto {loop_label}')
            CmdLst = [loop_stmnt] + L1 + [if_stmnt] + L2 + [goto_stmnt, next_stmnt]
            return CmdLst, max(sz1, sz2)
        case 'for', var, lower, upper, body_stmnt:
            L1, sz1    = compile_expr(lower, st, class_name)
            L2, sz2    = compile_expr(upper, st, class_name)
            L3, sz3    = compile_stmnt(body_stmnt, st, class_name)
            loop_label = new_label()
            next_label = new_label()
            loop_stmnt = ' ' * 4 + loop_label + ':'
            next_stmnt = ' ' * 4 + next_label + ':'
            goto_stmnt = indent(f'goto {loop_label}')
            istore_stmnt = indent(f'istore {st[var]}')
            iload_stmnt  = indent(f'iload  {st[var]}')
            iinc_stmnt   = indent(f'iinc {st[var]} 1')
            cmp_stmnt    = indent(f'if_icmpgt {next_label}')
            CmdLst = L1 + [istore_stmnt, loop_stmnt, iload_stmnt] + L2 + [cmp_stmnt] + \
                     L3 + [iinc_stmnt, goto_stmnt, next_stmnt]
            return CmdLst, max(2, sz1, sz2 + 1, sz3)
        case 'return', expr:
            CmdLst, sz = compile_expr(expr, st, class_name)
            CmdLst    += [indent('ireturn')]
            return CmdLst, sz
        case '.', *stmnt_lst:
            CmdLst = []
            size   = 0
            for s in stmnt_lst:
                L, sz = compile_stmnt(s, st, class_name)
                CmdLst += L
                size   = max(size, sz)
            return CmdLst, size
        case _: # it must be an expression statement
            return compile_expr(stmnt, st, class_name)

In [ ]:
stmnt = ('=', 'x', ('Number', '42'))
st    = { 'x': 0, 'y': 1}
compile_stmnt(stmnt, st, 'Sum')

In [ ]:
stmnt = ('if', ('==', 'x', 'y'), ('=', 'x', ('Number', '42')))
compile_stmnt(stmnt, st, 'Sum')

In [ ]:
stmnt = ('if-else', ('<', 'x', 'y'), ('=', 'x', 'y'), ('=', 'y', 'x'))
compile_stmnt(stmnt, st, 'Sum')

In [ ]:
stmnt = ('while', ('<', 'x', 'y'), ('=', 'x', ('+', 'x', ('Number', '1'))))
compile_stmnt(stmnt, st, 'Sum')

In [ ]:
stmnt = ('for', 'i', ('Number', '1'), 'n', ('=', 's', ('+', 's', 'i')))
st    = { 'i': 0, 'n': 1, 's': 2 }
compile_stmnt(stmnt, st, 'Sum')

In [ ]:
stmnt = ('.', ('=', 'x', 'y'), ('=', 'x', ('Number', '1')), ('=', 'y', 'x'))
st    = { 'x': 0, 'y': 1 }
compile_stmnt(stmnt, st, 'Sum')

In [ ]:
def compile_fct(fct_def, class_name):
    global label_counter
    label_counter = 0
    _, name, parameters, variables, stmnts = fct_def
    _, *parameters = parameters
    _, *variables  = variables
    _, *stmnts     = stmnts
    m   = len(parameters)
    n   = len(variables)
    st  = {}
    cnt = 0
    for var in parameters + variables:
        st[var] = cnt
        cnt    += 1
    CmdLst = []
    size   = 0
    for stmnt in stmnts:
        L, sz = compile_stmnt(stmnt, st, class_name)
        CmdLst += L
        size = max(size, sz)
    limit_locals = f'.limit locals {m+n}'
    limit_stack  = f'.limit stack  {size}'
    return_stmnt = indent('return')
    if name != 'main':
        method = f'.method public static {name}({"I"*m})I'
        CmdLst = [method, limit_locals, limit_stack] + CmdLst + ['.end method']
        return CmdLst, size
    else:
        method = '.method public static main([Ljava/lang/String;)V'
        CmdLst = [method, limit_locals, limit_stack] + CmdLst + [return_stmnt, '.end method']
        return CmdLst, size

In [ ]:
f = ('fct', 'sum', ('.', 'x'), ('.', 'y', 'z'), ('.', ('return', 'x')))
compile_fct(f, 'Sum')

In [ ]:
import os

In [ ]:
file = "~/Drive/Kurse/Formal-Languages/Python/Chapter-13/Examples/Primes.c"
print(os.path.dirname(file))
print(os.path.basename(file))
print(os.path.join('abc', 'xyz.c'))

The input to `compile_program(file_name)` is the name of a `C` file.  This file assumed to have the ending `.c`.
This file is compiled and the resulting assembler file is written into a file with the same name but the ending `.jas`.

In [ ]:
def compile_program(file_name):
    directory = os.path.dirname(file_name)
    base      = os.path.basename(file_name)
    base      = base[:-2]
    outfile   = os.path.join(directory, base + '.jas')
    with open(file_name, 'r') as handle:
        program = handle.read()
    ast = parse_program(program)
    if ast is None:
        return
    _, *fct_lst = ast
    CmdLst = []
    for fct in fct_lst:
        L, _ = compile_fct(fct, base)
        CmdLst += L + ['\n']
    with open(outfile, 'w') as handle:
        handle.write('.class public ' + base + '\n');
        handle.write('.super java/lang/Object\n\n');
        handle.write('.method public <init>()V\n');
        handle.write('    aload 0\n');
        handle.write('    invokenonvirtual java/lang/Object/<init>()V\n');
        handle.write('    return\n');
        handle.write('.end method\n\n');
        for cmd in CmdLst:
            handle.write(cmd + '\n')

The file `PrimesFor.c` is a simple `C` program that computes the prime numbers in a naive way using
`for` loops.

In [ ]:
%cd Examples

In [ ]:
!ls -l

In [ ]:
!cat PrimesFor.c

In [ ]:
compile_program('PrimesFor.c')

In [ ]:
!ls -l 

In [ ]:
!cat PrimesFor.jas

Next, we generate Java byte code using
[jasmin](http://jasmin.sourceforge.net).

In [ ]:
!jasmin PrimesFor.jas

Finally, we run the generated byte code.

In [ ]:
!ls -l

In [ ]:
!java PrimesFor